#  Bitcoin Trader Behavior vs Market Sentiment
## Primetrade.ai — Junior Data Scientist Assignment

**Objective:** Explore the relationship between trader performance (Hyperliquid) and Bitcoin market sentiment (Fear & Greed Index), uncover hidden patterns, and deliver actionable trading strategy insights.

---
**Datasets:**
- `historical_data.csv` — 211,224 trades from Hyperliquid
- `fear_greed_index.csv` — 2,644 daily Bitcoin sentiment scores (2018–2025)

**Date range after merge:** May 2023 – May 2025

In [ ]:
# ── Imports & Setup ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'

PALETTE = {
    'Extreme Fear':'#d62728','Fear':'#ff7f0e',
    'Neutral':'#8c8c8c','Greed':'#2ca02c','Extreme Greed':'#1f77b4'
}
SENT_ORDER = ['Extreme Fear','Fear','Neutral','Greed','Extreme Greed']
print(' Libraries loaded')

In [ ]:
# ── Load Datasets ─────────────────────────────────────────────────────────────
# Update paths if running locally
trader = pd.read_csv('historical_data.csv')
fg     = pd.read_csv('fear_greed_index.csv')

print(f'Trader data shape : {trader.shape}')
print(f'Fear/Greed shape  : {fg.shape}')
trader.head(3)

In [ ]:
# ── Data Cleaning & Merging ───────────────────────────────────────────────────
trader['date']      = pd.to_datetime(trader['Timestamp IST'], dayfirst=True).dt.date.astype(str)
trader['Closed PnL']= pd.to_numeric(trader['Closed PnL'], errors='coerce').fillna(0)
trader['Size USD']  = pd.to_numeric(trader['Size USD'],   errors='coerce').fillna(0)
trader['Fee']       = pd.to_numeric(trader['Fee'],        errors='coerce').fillna(0)

fg['date']  = pd.to_datetime(fg['date']).dt.date.astype(str)
fg['value'] = pd.to_numeric(fg['value'], errors='coerce')

merged = trader.merge(fg[['date','value','classification']], on='date', how='inner')
print(f'Merged rows   : {len(merged):,}')
print(f'Date range    : {merged["date"].min()} → {merged["date"].max()}')
print(f'Unique traders: {merged["Account"].nunique():,}')
print(f'Unique coins  : {merged["Coin"].nunique():,}')
merged.head(3)

In [ ]:
# ── EDA: Basic Stats ──────────────────────────────────────────────────────────
print('=== Sentiment Distribution ===')
print(merged['classification'].value_counts())

print('\n=== Closed PnL Summary ===')
print(merged['Closed PnL'].describe())

print('\n=== Side Distribution ===')
print(merged['Side'].value_counts())

In [ ]:
# ── Aggregations ──────────────────────────────────────────────────────────────
# Daily
daily = merged.groupby(['date','classification','value']).agg(
    total_pnl  = ('Closed PnL','sum'),
    avg_pnl    = ('Closed PnL','mean'),
    trades     = ('Closed PnL','count'),
    winners    = ('Closed PnL', lambda x: (x>0).sum()),
    volume_usd = ('Size USD','sum'),
    total_fee  = ('Fee','sum'),
).reset_index()
daily['win_rate'] = daily['winners'] / daily['trades'] * 100
daily['net_pnl']  = daily['total_pnl'] - daily['total_fee']

# Sentiment
sent = merged.groupby('classification').agg(
    total_pnl  = ('Closed PnL','sum'),
    avg_pnl    = ('Closed PnL','mean'),
    trades     = ('Closed PnL','count'),
    winners    = ('Closed PnL', lambda x: (x>0).sum()),
    volume_usd = ('Size USD','sum'),
).reset_index()
sent['win_rate'] = sent['winners'] / sent['trades'] * 100
sent = sent.set_index('classification').reindex(SENT_ORDER).reset_index()

print(sent[['classification','avg_pnl','win_rate','trades']].to_string(index=False))

In [ ]:
# ── Figure 1: Main Dashboard ──────────────────────────────────────────────────
colors = [PALETTE.get(c,'#999') for c in sent['classification']]

fig = plt.figure(figsize=(20,22), facecolor='#0d1117')
fig.suptitle('Bitcoin Trader Behavior vs Market Sentiment\nPrimetrade.ai – Data Science Assignment',
             fontsize=22, fontweight='bold', color='white', y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# Avg PnL per sentiment
ax1 = fig.add_subplot(gs[0,0])
bars = ax1.bar(sent['classification'], sent['avg_pnl'], color=colors, edgecolor='white', linewidth=0.5)
ax1.set_facecolor('#161b22'); ax1.set_title('Avg PnL per Trade by Sentiment', color='white', fontweight='bold')
ax1.set_ylabel('Avg Closed PnL (USD)', color='white'); ax1.tick_params(colors='white', rotation=20)
ax1.spines[:].set_color('#30363d')
for bar, val in zip(bars, sent['avg_pnl']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'${val:.1f}', ha='center', color='white', fontsize=9)

# Win Rate per sentiment
ax2 = fig.add_subplot(gs[0,1])
bars2 = ax2.bar(sent['classification'], sent['win_rate'], color=colors, edgecolor='white', linewidth=0.5)
ax2.set_facecolor('#161b22'); ax2.set_title('Win Rate (%) by Sentiment', color='white', fontweight='bold')
ax2.set_ylabel('Win Rate (%)', color='white'); ax2.tick_params(colors='white', rotation=20)
ax2.spines[:].set_color('#30363d'); ax2.axhline(50, color='yellow', linestyle='--', linewidth=1, alpha=0.7)
for bar, val in zip(bars2, sent['win_rate']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val:.1f}%', ha='center', color='white', fontsize=9)

# Volume by Sentiment
ax3 = fig.add_subplot(gs[0,2])
ax3.bar(sent['classification'], sent['volume_usd']/1e6, color=colors, edgecolor='white', linewidth=0.5)
ax3.set_facecolor('#161b22'); ax3.set_title('Total Trade Volume by Sentiment', color='white', fontweight='bold')
ax3.set_ylabel('Volume (Million USD)', color='white'); ax3.tick_params(colors='white', rotation=20)
ax3.spines[:].set_color('#30363d')

# Daily PnL scatter
ax4 = fig.add_subplot(gs[1,:])
for cls in SENT_ORDER:
    sub = daily[daily['classification']==cls]
    ax4.scatter(sub['date'], sub['net_pnl'], label=cls, color=PALETTE[cls], alpha=0.6, s=20)
ax4.set_facecolor('#161b22'); ax4.set_title('Daily Net PnL Over Time (colored by Sentiment)', color='white', fontweight='bold')
ax4.set_ylabel('Net PnL (USD)', color='white'); ax4.tick_params(colors='white')
ax4.spines[:].set_color('#30363d'); ax4.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax4.legend(facecolor='#161b22', labelcolor='white', fontsize=9)
dates = daily['date'].tolist(); step = max(1, len(dates)//10)
ax4.set_xticks(range(0, len(dates), step))
ax4.set_xticklabels([dates[i] for i in range(0, len(dates), step)], rotation=30, fontsize=8)

# Scatter: sentiment value vs avg PnL
ax5 = fig.add_subplot(gs[2,0])
sc = ax5.scatter(daily['value'], daily['avg_pnl'], c=daily['value'], cmap='RdYlGn', alpha=0.7, s=25)
plt.colorbar(sc, ax=ax5, label='Fear/Greed Index')
m, b, r, p, _ = stats.linregress(daily['value'], daily['avg_pnl'])
x_line = np.linspace(daily['value'].min(), daily['value'].max(), 100)
ax5.plot(x_line, m*x_line+b, 'yellow', linewidth=1.5, linestyle='--')
ax5.set_facecolor('#161b22'); ax5.set_title(f'Sentiment Index vs Avg PnL (r={r:.3f})', color='white', fontweight='bold')
ax5.set_xlabel('Fear/Greed Value', color='white'); ax5.set_ylabel('Avg PnL (USD)', color='white')
ax5.tick_params(colors='white'); ax5.spines[:].set_color('#30363d')

# BUY vs SELL by sentiment
side_sent = merged.groupby(['classification','Side']).agg(avg_pnl=('Closed PnL','mean')).reset_index()
ax6 = fig.add_subplot(gs[2,1])
buy_data  = side_sent[side_sent['Side']=='BUY'].set_index('classification').reindex(SENT_ORDER)['avg_pnl']
sell_data = side_sent[side_sent['Side']=='SELL'].set_index('classification').reindex(SENT_ORDER)['avg_pnl']
x = np.arange(len(SENT_ORDER)); w = 0.35
ax6.bar(x-w/2, buy_data.values,  w, label='BUY',  color='#2ca02c', edgecolor='white', linewidth=0.5)
ax6.bar(x+w/2, sell_data.values, w, label='SELL', color='#d62728', edgecolor='white', linewidth=0.5)
ax6.set_facecolor('#161b22'); ax6.set_title('Avg PnL: BUY vs SELL by Sentiment', color='white', fontweight='bold')
ax6.set_xticks(x); ax6.set_xticklabels(SENT_ORDER, rotation=20, color='white', fontsize=8)
ax6.set_ylabel('Avg PnL (USD)', color='white'); ax6.tick_params(colors='white')
ax6.spines[:].set_color('#30363d'); ax6.legend(facecolor='#161b22', labelcolor='white')
ax6.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)

# Pie chart
ax7 = fig.add_subplot(gs[2,2])
ax7.set_facecolor('#161b22')
wedges, texts, autotexts = ax7.pie(
    sent['trades'], labels=sent['classification'], colors=colors,
    autopct='%1.1f%%', startangle=90, textprops={'color':'white','fontsize':9})
for at in autotexts: at.set_color('white')
ax7.set_title('Trade Distribution by Sentiment', color='white', fontweight='bold')

# Top 10 traders
top_traders = merged.groupby('Account').agg(
    total_pnl=('Closed PnL','sum'), trades=('Closed PnL','count'),
    win_rate=('Closed PnL', lambda x: (x>0).mean()*100)
).reset_index().sort_values('total_pnl', ascending=False).head(10)
top_traders['short'] = top_traders['Account'].str[:10]+'...'
ax8 = fig.add_subplot(gs[3,:2])
c_top = ['#2ca02c' if v>0 else '#d62728' for v in top_traders['total_pnl']]
ax8.barh(top_traders['short'], top_traders['total_pnl'], color=c_top, edgecolor='white', linewidth=0.5)
ax8.set_facecolor('#161b22'); ax8.set_title('Top 10 Traders by Total PnL', color='white', fontweight='bold')
ax8.set_xlabel('Total Closed PnL (USD)', color='white')
ax8.tick_params(colors='white'); ax8.spines[:].set_color('#30363d'); ax8.axvline(0, color='white', linewidth=0.8)

# Top coins
coin_perf = merged.groupby('Coin').agg(total_pnl=('Closed PnL','sum'), trades=('Closed PnL','count')
).reset_index().sort_values('total_pnl', ascending=False).head(10)
ax9 = fig.add_subplot(gs[3,2])
c_coin = ['#2ca02c' if v>0 else '#d62728' for v in coin_perf['total_pnl']]
ax9.barh(coin_perf['Coin'], coin_perf['total_pnl'], color=c_coin, edgecolor='white', linewidth=0.5)
ax9.set_facecolor('#161b22'); ax9.set_title('Top 10 Coins by PnL', color='white', fontweight='bold')
ax9.set_xlabel('Total PnL (USD)', color='white')
ax9.tick_params(colors='white'); ax9.spines[:].set_color('#30363d')

plt.savefig('analysis_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(' Dashboard saved')

In [ ]:
# ── Figure 2: Deep Dive ───────────────────────────────────────────────────────
fig2, axes = plt.subplots(2,2, figsize=(18,12), facecolor='#0d1117')
fig2.suptitle('Deep Dive: Hidden Patterns in Trader Behavior', fontsize=18, fontweight='bold', color='white')

# Box: PnL distribution per sentiment
ax = axes[0,0]; ax.set_facecolor('#161b22')
data_box = [merged[merged['classification']==c]['Closed PnL'].clip(-500,500) for c in SENT_ORDER]
bp = ax.boxplot(data_box, labels=SENT_ORDER, patch_artist=True, medianprops=dict(color='yellow',linewidth=2))
for patch, color in zip(bp['boxes'], [PALETTE[c] for c in SENT_ORDER]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('PnL Distribution by Sentiment (clipped ±500)', color='white', fontweight='bold')
ax.set_ylabel('Closed PnL (USD)', color='white'); ax.tick_params(colors='white', rotation=20)
ax.spines[:].set_color('#30363d'); ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)

# Rolling 7-day PnL
ax = axes[0,1]; ax.set_facecolor('#161b22')
daily_sorted = daily.sort_values('date')
daily_sorted['roll7'] = daily_sorted['avg_pnl'].rolling(7, min_periods=1).mean()
ax.plot(range(len(daily_sorted)), daily_sorted['roll7'], color='#1f77b4', linewidth=1.5)
ax.fill_between(range(len(daily_sorted)), daily_sorted['roll7'], alpha=0.2, color='#1f77b4')
ax.axhline(0, color='white', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('7-Day Rolling Avg PnL Over Time', color='white', fontweight='bold')
ax.set_ylabel('Avg PnL (USD)', color='white'); ax.tick_params(colors='white'); ax.spines[:].set_color('#30363d')

# Heatmap: Win rate by sentiment x side
ax = axes[1,0]; ax.set_facecolor('#161b22')
heat_data = merged.groupby(['classification','Side'])['Closed PnL'].apply(lambda x: (x>0).mean()*100).unstack()
heat_data = heat_data.reindex(SENT_ORDER)
sns.heatmap(heat_data, ax=ax, cmap='RdYlGn', annot=True, fmt='.1f', linewidths=0.5, cbar_kws={'label':'Win Rate %'})
ax.set_title('Win Rate Heatmap: Sentiment × Side', color='white', fontweight='bold')
ax.tick_params(colors='white')

# Stacked area: volume by sentiment
ax = axes[1,1]; ax.set_facecolor('#161b22')
vol_sent = merged.groupby(['date','classification'])['Size USD'].sum().unstack(fill_value=0)
vol_sent = vol_sent.reindex(columns=SENT_ORDER, fill_value=0)
bottom = np.zeros(len(vol_sent))
for cls in SENT_ORDER:
    if cls in vol_sent.columns:
        vals = vol_sent[cls].values / 1e6
        ax.fill_between(range(len(vol_sent)), bottom, bottom+vals, alpha=0.7, label=cls, color=PALETTE[cls])
        bottom += vals
ax.set_title('Daily Volume by Sentiment (Stacked)', color='white', fontweight='bold')
ax.set_ylabel('Volume (M USD)', color='white'); ax.tick_params(colors='white')
ax.spines[:].set_color('#30363d'); ax.legend(facecolor='#161b22', labelcolor='white', fontsize=9)

plt.tight_layout()
plt.savefig('deep_dive.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(' Deep dive saved')

##  Key Insights & Strategic Recommendations

### 1. Extreme Greed = Highest Avg PnL ($67.9/trade)
Traders earn the most per trade during **Extreme Greed** periods. This suggests that trend-following strategies (buying during strong bull momentum) are most profitable.

### 2. Win Rate is Highest in Extreme Greed (46.5%)
Despite no sentiment zone crossing 50% win rate, **Extreme Greed** has the highest win rate. This aligns with the classic 'trend is your friend' principle — markets in euphoria tend to continue upward.

### 3. Fear Zones Have Higher Avg PnL Than Neutral
Avg PnL during **Fear ($54.3)** is higher than **Neutral ($34.3)** and even **Greed ($42.7)**. This points to contrarian opportunity — experienced traders profit from fear by buying dips.

### 4. Weak Correlation Between Sentiment Index and PnL (r=0.037)
The Pearson correlation is very low and statistically insignificant (p=0.41). This means **raw sentiment score alone is not a strong PnL predictor** — classification categories matter more than the numeric score.

### 5. Most Trading Happens During Fear (61,837 trades)
The highest trade count occurs in **Fear** periods, meaning traders are most active when markets are uncertain — possibly due to increased volatility creating more opportunities.

### 6. Extreme Greed has highest Volume despite fewer days
Despite fewer trading days in Extreme Greed, trade volume remains high, confirming that bull markets drive aggressive position sizing.

---

##  Strategic Recommendations

| Strategy | Sentiment Zone | Action |
|---|---|---|
| Trend Following | Extreme Greed | Long positions, ride momentum |
| Contrarian Buy | Extreme Fear | Accumulate — highest risk-reward |
| Reduce Exposure | Neutral | Smaller positions, await direction |
| Selective Short | Greed → Fear transition | Watch for reversals |
| Capital Preservation | Extreme Fear (first sign) | Tighten stops |

---
*Analysis by: [Your Name] | Dataset: Hyperliquid + Alternative.me Fear & Greed Index*